# 01 · CMIP6 large-ensemble FWI for India
**Project:** Unprecedented pre-monsoon fire weather over Central India and Uttarakhand, today and at 1.5 / 2 / 3 °C.

This notebook downloads the **pre-computed CMIP6 Fire Weather Index annual indicators** of Quilcaille et al. (2023, ESSD,
doi:10.5194/essd-15-2153-2023; data doi:10.3929/ethz-b-000583391, CC-BY 4.0) and keeps only:

| Single-model large ensemble | members (FWI from daily tasmax + **hursmin** + sfcWind + pr) |
|---|---|
| CanESM5 | 50 |
| MIROC6 | 50 |
| ACCESS-ESM1-5 | 40 |
| MPI-ESM1-2-LR | 30 |

experiments `historical` (1850–2014) + `ssp585` (2015–2100), over an India box on the dataset's common 2.5° grid.
Indicators: **fwixx** (annual max FWI), **fwisa** (annual peak of 90-day mean FWI), **fwixd** (days ≥ local 95th pct of 1850–1899), **fwils** (fire-season length).

It also builds the **global-warming-level (GWL) windows** for every member (Hauser et al., doi:10.5281/zenodo.3591806):
20-year windows centred on the year each member reaches 1.3 °C ("today"), 1.5, 2 and 3 °C above 1850–1900.

**Run this notebook first.** Runtime ≈ 30–60 min (mostly downloading zips to Colab's local disk; only ~50 MB is written to Drive).
Outputs → `MyDrive/FWI_India/data/cmip6/`.

In [ ]:
# ================= CONFIG =================
import os
BASE = os.environ.get('FWI_BASE', '/content/drive/MyDrive/FWI_India')
OUT = f'{BASE}/data/cmip6'
TMP = os.environ.get('FWI_TMP', '/content/tmp_zips')          # Colab local disk (ephemeral, ~80–100 GB free)

MODELS = ['CanESM5', 'MIROC6', 'ACCESS-ESM1-5', 'MPI-ESM1-2-LR']
EXPS = ['historical', 'ssp585']
INDICATORS = ['fwixx', 'fwisa', 'fwixd', 'fwils']
HUMIDITY = 'hursmin'                       # 'hursmin' = main dataset (daily MIN RH) – matches our Tmax/RHmin proxy
DOMAIN = dict(lat=(4.0, 39.0), lon=(66.0, 100.0))   # generous box around India (trimmed to whole cells)

GWLS = {'today': 1.3, '1.5': 1.5, '2.0': 2.0, '3.0': 3.0}  # 'today' interpolated between 1.2 and 1.5 °C crossings
GWL_SCENARIO = 'ssp585'
GWL_CSV = ('https://raw.githubusercontent.com/mathause/cmip_warming_levels/main/'
           'warming_levels/cmip6_all_ens/csv/cmip6_warming_levels_all_ens_1850_1900.csv')

RC = 'https://www.research-collection.ethz.ch'
ITEM_UUID = '1b12cf29-794a-4987-b455-6164ba5c4294'
# Fallback if the repository API is blocked: paste download links here, e.g.
# MANUAL_URLS = {'fwixd': ['https://www.research-collection.ethz.ch/bitstreams/b5c52259-0adf-449a-a309-d47568a04e80/download'], ...}
MANUAL_URLS = {}

In [ ]:
# [COLAB-ONLY]
from google.colab import drive
drive.mount('/content/drive')
!pip -q install cftime netCDF4      # needed to read the CMIP6 time axes (365-day / proleptic calendars)

In [ ]:
import re, io, json, glob, time, zipfile, shutil, warnings
import numpy as np, pandas as pd, xarray as xr, requests
import cftime  # noqa: F401  (fails early if the install above was skipped)
warnings.filterwarnings('ignore')
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
print('Output folder:', OUT)

## 1. Find the right zip files in the ETH repository
The repository lists ~20+ zip archives (4 indicators × 2 humidity variants, some split in parts). We query its public API and pick
the **hursmin** archives of the four indicators. Check the printed table — if a needed file is not recognised, put its link in `MANUAL_URLS`.

In [ ]:
def list_bitstreams():
    s = requests.Session(); s.headers['User-Agent'] = 'Mozilla/5.0 (FWI-India research notebook)'
    r = s.get(f'{RC}/server/api/core/items/{ITEM_UUID}/bundles', params={'size': 50}, timeout=60)
    r.raise_for_status()
    rows = []
    for b in r.json()['_embedded']['bundles']:
        if b['name'] != 'ORIGINAL':
            continue
        href, page = b['_links']['bitstreams']['href'], 0
        while True:
            j = s.get(href, params={'size': 100, 'page': page}, timeout=60).json()
            for bs in j['_embedded']['bitstreams']:
                md = bs.get('metadata', {})
                desc = ' '.join(m['value'] for k in ('dc.description', 'dc.title') for m in md.get(k, []))
                rows.append(dict(name=bs['name'], desc=desc, size_mb=round(bs['sizeBytes'] / 1e6, 1),
                                 url=f"{RC}/server/api/core/bitstreams/{bs['uuid']}/content"))
            if page + 1 >= j['page']['totalPages']:
                break
            page += 1
    return pd.DataFrame(rows)

def classify(name, desc):
    t = f'{name} {desc}'.lower()
    if 'hursmin' in t or 'minimum relative humidity' in t:
        hum = 'hursmin'
    elif re.search(r'hurs(?!min)', t) or 'average relative humidity' in t or 'mean relative humidity' in t:
        hum = 'hurs'
    else:
        hum = '?'
    if 'fwixx' in t or re.search(r'maxim\w* (value )?of the (fire weather index|fwi)', t) or 'annual maximum' in t:
        ind = 'fwixx'
    elif 'fwisa' in t or 'seasonal average' in t:
        ind = 'fwisa'
    elif 'fwixd' in t or 'number of days' in t:
        ind = 'fwixd'
    elif 'fwils' in t or 'length of the fire season' in t:
        ind = 'fwils'
    else:
        ind = '?'
    return ind, hum

In [ ]:
# [COLAB-ONLY]
try:
    files = list_bitstreams()
    files[['indicator', 'humidity']] = [classify(n, d) for n, d in zip(files.name, files.desc)]
    pd.set_option('display.max_colwidth', 90)
    display(files[['name', 'indicator', 'humidity', 'size_mb', 'desc']])
except Exception as e:
    print('Repository API failed:', e, '\n→ Fill MANUAL_URLS in the CONFIG cell with the hursmin links from the web page.')
    files = pd.DataFrame(columns=['name', 'desc', 'size_mb', 'url', 'indicator', 'humidity'])

TODO = {ind: [] for ind in INDICATORS}
for ind in INDICATORS:
    if MANUAL_URLS.get(ind):
        TODO[ind] = list(MANUAL_URLS[ind])
    else:
        TODO[ind] = files[(files.indicator == ind) & (files.humidity == HUMIDITY)].url.tolist()
for ind, urls in TODO.items():
    print(f'{ind}: {len(urls)} archive(s)')
assert all(TODO.values()), 'Some indicators have no archive → set MANUAL_URLS (see table above).'
print('Total download ≈', files[files.url.isin(sum(TODO.values(), []))].size_mb.sum() / 1e3, 'GB (temporary, local disk)')

## 2. Download → extract only our models/experiments → cut out India
Each archive is streamed to Colab's local disk, the matching NetCDF files are opened one by one, subset to the India box, and the zip is deleted.

In [ ]:
FNAME_RE = re.compile(r'^(fwi[a-z]{2})_ann_(.+?)_(historical|ssp\d{3}(?:-over)?)_(r\d+i\d+p\d+f\d+)_(g\d+)\.nc$')

def download(url, dst, tries=10):
    """Resumable download; waits as asked on HTTP 429 (server rate limit)."""
    if os.path.exists(dst) and zipfile.is_zipfile(dst):
        print('  zip already on disk, reusing'); return dst
    for k in range(tries):
        try:
            have = os.path.getsize(dst) if os.path.exists(dst) else 0
            hdr = {'User-Agent': 'Mozilla/5.0 (FWI-India research notebook)'}
            if have:
                hdr['Range'] = f'bytes={have}-'
            with requests.get(url, stream=True, timeout=180, headers=hdr) as r:
                if r.status_code == 429:
                    wait = int(r.headers.get('Retry-After', 0) or 0) or 60 * (k + 1)
                    print(f'\n  server busy (429) → waiting {wait} s'); time.sleep(wait); continue
                if have and r.status_code != 206:      # server ignored Range → restart
                    have = 0
                r.raise_for_status()
                tot = have + int(r.headers.get('content-length', 0) or 0); got = have; t0 = time.time()
                with open(dst, 'ab' if have else 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk); got += len(chunk)
                        tt = f'{tot/1e9:5.2f}' if tot > have else '  ?  '
                        print(f'\r  {got/1e9:5.2f} / {tt} GB  ({(got-have)/1e6/max(time.time()-t0,1e-3):.0f} MB/s)', end='')
            print()
            if zipfile.is_zipfile(dst):
                return dst
            print('  incomplete zip, resuming …')
        except Exception as e:
            print(f'\n  retry {k+1}/{tries}: {e}'); time.sleep(30 * (k + 1))
    raise RuntimeError(f'Download failed: {url}')

def subset_india(ds, var):
    da = ds[var]
    lat = [d for d in da.dims if d.lower().startswith('lat')][0]
    lon = [d for d in da.dims if d.lower().startswith('lon')][0]
    da = da.rename({lat: 'lat', lon: 'lon'}).sortby('lat').sortby('lon')
    if float(da.lon.max()) > 180 and DOMAIN['lon'][0] < 0:
        da = da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby('lon')
    da = da.sel(lat=slice(*DOMAIN['lat']), lon=slice(*DOMAIN['lon']))
    years = da['time'].dt.year.values
    return da.assign_coords(time=years).rename(time='year').astype('float32').load()

def process_zip(path, ind, store):
    n = 0
    with zipfile.ZipFile(path) as zf:
        for info in zf.infolist():
            m = FNAME_RE.match(os.path.basename(info.filename))
            if not m:
                continue
            indic, esm, xp, mem, grid = m.groups()
            if indic != ind or esm not in MODELS or xp not in EXPS:
                continue
            tmp_nc = os.path.join(TMP, os.path.basename(info.filename))
            with zf.open(info) as src, open(tmp_nc, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            with xr.open_dataset(tmp_nc, use_cftime=True) as ds:
                store[(esm, mem, xp)] = subset_india(ds, indic)
            os.remove(tmp_nc); n += 1
    return n

In [ ]:
# [COLAB-ONLY]
PARTS = f'{OUT}/_parts'; os.makedirs(PARTS, exist_ok=True)     # per-archive checkpoints (small, on Drive)
for ind in INDICATORS:
    out_nc = f'{OUT}/LE_{ind}_{HUMIDITY}_india.nc'
    if os.path.exists(out_nc):
        print(ind, '→ already done, skipping'); continue
    for k, url in enumerate(TODO[ind]):
        part = f'{PARTS}/{ind}_{HUMIDITY}_{k}.pkl'
        if os.path.exists(part):
            print(f'[{ind}] archive {k+1}/{len(TODO[ind])} already processed'); continue
        zpath = os.path.join(TMP, f'{ind}_{HUMIDITY}_{k}.zip')
        print(f'[{ind}] archive {k+1}/{len(TODO[ind])}')
        download(url, zpath)
        store = {}
        print('   extracted', process_zip(zpath, ind, store), 'files')
        pd.to_pickle(store, part)
        os.remove(zpath)
        time.sleep(20)                                         # be gentle with the repository (avoids 429)
    store = {}
    for f in sorted(glob.glob(f'{PARTS}/{ind}_{HUMIDITY}_*.pkl')):
        store.update(pd.read_pickle(f))
    pd.to_pickle(store, f'{TMP}/{ind}_store.pkl')
    print(ind, 'runs kept:', len(store))

## 3. Join historical + ssp585 per member → one file per indicator `(run, year, lat, lon)`

In [ ]:
def assemble(store, ind):
    keys = sorted({(e, m) for (e, m, x) in store})
    ref = next(iter(store.values()))
    years = np.arange(1850, 2101)
    arr = np.full((len(keys), years.size, ref.lat.size, ref.lon.size), np.nan, dtype='float32')
    for i, (e, m) in enumerate(keys):
        for xp in EXPS:
            da = store.get((e, m, xp))
            if da is None:
                continue
            yy = da.year.values
            sel = (yy >= years[0]) & (yy <= years[-1])
            if xp == 'historical':
                sel &= yy <= 2014
            else:
                sel &= yy >= 2015
            arr[i, yy[sel] - years[0]] = da.values[sel]
    da = xr.DataArray(arr, dims=('run', 'year', 'lat', 'lon'),
                      coords=dict(run=[f'{e}|{m}' for e, m in keys], year=years, lat=ref.lat.values, lon=ref.lon.values,
                                  esm=('run', [e for e, m in keys]), member=('run', [m for e, m in keys])),
                      name=ind, attrs=dict(source='Quilcaille et al. 2023 ESSD, doi:10.3929/ethz-b-000583391 (CC-BY 4.0)',
                                           humidity=HUMIDITY, experiments='historical 1850-2014 + ssp585 2015-2100'))
    has_hist = np.isfinite(arr[:, 150]).any(axis=(1, 2))      # 2000
    has_fut = np.isfinite(arr[:, 220]).any(axis=(1, 2))       # 2070
    da = da.assign_coords(has_hist=('run', has_hist), has_ssp585=('run', has_fut))
    return da

In [ ]:
for ind in INDICATORS:
    out_nc = f'{OUT}/LE_{ind}_{HUMIDITY}_india.nc'
    if os.path.exists(out_nc):
        continue
    da = assemble(pd.read_pickle(f'{TMP}/{ind}_store.pkl'), ind)
    da.to_netcdf(out_nc, encoding={ind: {'zlib': True, 'complevel': 4}})
    print('saved', out_nc, f'{os.path.getsize(out_nc)/1e6:.1f} MB')

## 4. Check what we have + record the 2.5° grid (used by notebook 02 to align ERA5)

In [ ]:
da = xr.open_dataarray(f'{OUT}/LE_fwixx_{HUMIDITY}_india.nc')
summary = pd.DataFrame({'historical': da.has_hist.groupby(da.esm).sum().to_series(),
                        'ssp585': da.has_ssp585.groupby(da.esm).sum().to_series(),
                        'both': (da.has_hist & da.has_ssp585).groupby(da.esm).sum().to_series()})
print(summary)
lat, lon = da.lat.values, da.lon.values
dlat, dlon = np.diff(lat).mean(), np.diff(lon).mean()
grid = dict(lat=lat.tolist(), lon=lon.tolist(), dlat=float(dlat), dlon=float(dlon),
            lat_edges=np.r_[lat - dlat / 2, lat[-1] + dlat / 2].tolist(),
            lon_edges=np.r_[lon - dlon / 2, lon[-1] + dlon / 2].tolist())
json.dump(grid, open(f'{BASE}/data/grid_g025.json', 'w'), indent=1)
print('grid spacing', dlat, dlon, '| first cell centre', lat[0], lon[0])
assert np.allclose(np.diff(lat), 2.5) and np.allclose(np.diff(lon), 2.5), 'unexpected grid'

## 5. Global-warming-level windows per member (20-yr windows, relative to 1850–1900)
Hauser et al. give crossing windows for 1.0, 1.2, 1.5, 2, 3, 4 °C. "Today" (≈1.3 °C, roughly the current level of
human-induced warming) is linearly interpolated between the 1.2 and 1.5 °C central years.

In [ ]:
def gwl_windows(csv_src, models=MODELS, scen=GWL_SCENARIO, gwls=GWLS):
    w = pd.read_csv(csv_src, comment='#', skipinitialspace=True)
    w.columns = [c.strip() for c in w.columns]
    w = w[(w.exp == scen) & w.model.isin(models)].copy()
    w['central'] = w.start_year + 10
    piv = w.pivot_table(index=['model', 'ensemble'], columns='warming_level', values='central')
    rows = []
    for (mod, mem), r in piv.iterrows():
        for name, g in gwls.items():
            if g in r.index and np.isfinite(r[g]):
                c = r[g]
            elif np.isfinite(r.get(1.2, np.nan)) and np.isfinite(r.get(1.5, np.nan)) and 1.2 < g < 1.5:
                c = r[1.2] + (r[1.5] - r[1.2]) * (g - 1.2) / 0.3
            else:
                continue
            c = int(round(c))
            rows.append(dict(esm=mod, member=mem, gwl=name, central=c, start=c - 10, end=c + 9))
    return pd.DataFrame(rows)

win = gwl_windows(GWL_CSV)
win = win[(win.end <= 2100)]
runs = pd.Index(da.run.values)
win['run'] = win.esm + '|' + win.member
win['in_fwi_data'] = win.run.isin(runs)
win.to_csv(f'{OUT}/gwl_windows_{GWL_SCENARIO}.csv', index=False)
print(win[win.in_fwi_data].groupby(['esm', 'gwl']).agg(members=('member', 'nunique'),
      median_central=('central', 'median')).unstack('gwl'))